# MTCA-1 Stage 3: Frame Design and Sampling Protocol

**Study identifier:** MTCA-1
**Stage:** 3 of 10 — Frame design, prompt construction, and sample selection
**Inputs:** Corpus v1.1 (frozen at Stage 2.5)
**Outputs:** Eight frozen frame prompts (SHA256-hashed), a deterministic 54-unit sample, and a frozen Stage 3 design artifact ready for Stage 5 pilot and Stage 6 execution

## Purpose

Define the eight framing conditions from the Stage 1 Foundational Document, lock them as frozen prompt templates with SHA256 hashes, draw the stratified sample (10 units per text or all units if text has ≤12), and freeze the complete Stage 3 design artifact.

**No council runs happen in this notebook.** Execution is Stage 5 (pilot) and Stage 6 (full).

## Scope (locked from prior conversation)

- 54 units total (10 from C1, C3, C5; all 12 from C7 and C8)
- 8 framing conditions
- 5 council models (Claude Sonnet 4.6, GPT, Gemini, Grok, DeepSeek)
- Total: 2,160 API calls for full execution
- Budget: ~$25-50 across all providers

## Section 1: Environment and corpus load

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount('/content/drive')
    except Exception:
        pass
    PROJECT_ROOT = '/content/drive/MyDrive/MTCA-1'
else:
    PROJECT_ROOT = './MTCA-1'

import os
import json
import hashlib
import random
from datetime import datetime, timezone
from dataclasses import dataclass, field, asdict

for subdir in ['stage3', 'stage3/frames', 'stage3/samples']:
    os.makedirs(f'{PROJECT_ROOT}/{subdir}', exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# Load corpus v1.1
frozen_dir = f'{PROJECT_ROOT}/corpus/frozen'
candidates = [f for f in os.listdir(frozen_dir) if f.startswith('corpus_v1.1_')]
if not candidates:
    raise FileNotFoundError('No corpus v1.1 found. Run Stage 2.5 cleanup first.')

corpus_path = f'{frozen_dir}/{candidates[0]}'
with open(corpus_path, 'r', encoding='utf-8') as f:
    corpus = json.load(f)

CORPUS_SHA = corpus['freeze_sha256']
CORPUS_VERSION = corpus['corpus_version']

print(f'Loaded {CORPUS_VERSION} from {corpus_path}')
print(f'Corpus SHA256: {CORPUS_SHA}')
print(f'Texts: {len(corpus["texts"])}')
print(f'Total units: {sum(len(t["units"]) for t in corpus["texts"])}')

## Section 2: Sampling protocol

Stratified sample: 10 units per text, OR all units if text has 12 or fewer. Random seed fixed at 42 for full reproducibility. The sampling function is deterministic — running it twice produces identical samples.

In [ ]:
SAMPLE_PER_TEXT = 10  # target units per text
SAMPLE_RANDOM_SEED = 42  # fixed for reproducibility

def stratified_sample(corpus, n_per_text=10, seed=42):
    """Deterministic stratified sample - same inputs always produce same output."""
    rng = random.Random(seed)
    sampled_units = []
    sample_meta = []
    
    for text in corpus['texts']:
        text_units = text['units']
        text_id = text['text_id']
        
        if len(text_units) <= n_per_text:
            # Take all units if text has fewer than target
            selected = list(text_units)
            sampling_method = 'all_units'
        else:
            # Random sample without replacement
            indices = rng.sample(range(len(text_units)), n_per_text)
            indices.sort()  # preserve document order in sample
            selected = [text_units[i] for i in indices]
            sampling_method = f'random_sample_seed_{seed}'
        
        sampled_units.extend(selected)
        sample_meta.append({
            'text_id': text_id,
            'text_title': text['title'],
            'lineage': text['lineage'],
            'units_in_corpus': len(text_units),
            'units_sampled': len(selected),
            'sampling_method': sampling_method,
            'sampled_unit_ids': [u['unit_id'] for u in selected]
        })
    
    return sampled_units, sample_meta

sampled_units, sample_meta = stratified_sample(corpus, SAMPLE_PER_TEXT, SAMPLE_RANDOM_SEED)

print(f'Stratified sample drawn (seed={SAMPLE_RANDOM_SEED}, target={SAMPLE_PER_TEXT}/text):')
print(f'{"text":4} {"available":>10} {"sampled":>8} {"method":>30}')
print('-' * 60)
for m in sample_meta:
    print(f'{m["text_id"]:4} {m["units_in_corpus"]:>10} {m["units_sampled"]:>8} {m["sampling_method"]:>30}')
print(f'\nTotal sampled units: {len(sampled_units)}')

In [ ]:
# Sanity check: inspect the actual sample
print('Sample preview (first 2 units per text):\n')
for text_id in ['C1', 'C3', 'C5', 'C7', 'C8']:
    text_sample = [u for u in sampled_units if u['text_id'] == text_id]
    print(f'\n=== {text_id} ({len(text_sample)} units) ===')
    for u in text_sample[:2]:
        print(f'  [{u["unit_id"]}] {u["unit_text"][:180]}')

## Section 3: Frame definitions

Eight framing conditions from the Stage 1 Foundational Document. Each frame is a templated prompt where `{UNIT}` is replaced by the unit text at runtime. For frames 7 and 8 (author identity), `{AUTHOR}` and `{TITLE}` are also templated.

Each prompt is frozen with SHA256. Any wording change creates a new version with a new hash.

**Design principle:** prompts ask the model for the same kind of structured response across all frames. Only the framing context varies. This is what makes Project B's cross-frame comparison meaningful — the response slot is constant; only the interpretive lens differs.

In [ ]:
# Frame definitions. Each frame includes:
# - frame_id: short identifier
# - name: human-readable name
# - description: what the frame does
# - prompt_template: the actual prompt sent to models, with {UNIT}/{AUTHOR}/{TITLE} substitution

FRAMES = {
    'F0_neutral': {
        'name': 'No frame (neutral baseline)',
        'description': 'Unit presented without any interpretive frame. Tests default model behavior.',
        'requires_author_metadata': False,
        'prompt_template': '''Read the following statement and respond with a brief structured analysis.

Statement: "{UNIT}"

Provide your response as JSON with the following keys:
- communicative_function: what the statement is doing communicatively
- implicit_claims: any claims the statement implies or presupposes (list)
- coherence_assessment: brief assessment of internal coherence
- conditions_for_usefulness: what would have to be true for this statement to be useful
- conditions_for_misleading: what would have to be true for this statement to be misleading

Respond in JSON only, no preamble.'''
    },
    
    'F1_clinical': {
        'name': 'Clinical therapeutic intention',
        'description': 'Frames the unit as a clinical therapeutic intention statement.',
        'requires_author_metadata': False,
        'prompt_template': '''The following statement is presented as a clinical therapeutic intention — a self-statement used in evidence-based psychotherapeutic practice.

Statement: "{UNIT}"

Under this clinical-therapeutic frame, provide your response as JSON with the following keys:
- communicative_function: what the statement is doing communicatively
- implicit_claims: any claims the statement implies or presupposes (list)
- coherence_assessment: brief assessment of internal coherence
- conditions_for_usefulness: what would have to be true for this statement to be useful
- conditions_for_misleading: what would have to be true for this statement to be misleading

Respond in JSON only, no preamble.'''
    },
    
    'F2_metaphysical': {
        'name': 'Metaphysical / spiritual assertion',
        'description': 'Frames the unit as a metaphysical or spiritual assertion.',
        'requires_author_metadata': False,
        'prompt_template': '''The following statement is presented as a metaphysical or spiritual assertion — a claim about non-physical reality, consciousness, or transcendent truth.

Statement: "{UNIT}"

Under this metaphysical-spiritual frame, provide your response as JSON with the following keys:
- communicative_function: what the statement is doing communicatively
- implicit_claims: any claims the statement implies or presupposes (list)
- coherence_assessment: brief assessment of internal coherence
- conditions_for_usefulness: what would have to be true for this statement to be useful
- conditions_for_misleading: what would have to be true for this statement to be misleading

Respond in JSON only, no preamble.'''
    },
    
    'F3_behavioral': {
        'name': 'Behavioral commitment',
        'description': 'Frames the unit as a behavioral commitment statement.',
        'requires_author_metadata': False,
        'prompt_template': '''The following statement is presented as a behavioral commitment — a statement of intended action or value-aligned behavior change.

Statement: "{UNIT}"

Under this behavioral-commitment frame, provide your response as JSON with the following keys:
- communicative_function: what the statement is doing communicatively
- implicit_claims: any claims the statement implies or presupposes (list)
- coherence_assessment: brief assessment of internal coherence
- conditions_for_usefulness: what would have to be true for this statement to be useful
- conditions_for_misleading: what would have to be true for this statement to be misleading

Respond in JSON only, no preamble.'''
    },
    
    'F4_poetic': {
        'name': 'Poetic expression',
        'description': 'Frames the unit as a poetic or literary expression.',
        'requires_author_metadata': False,
        'prompt_template': '''The following statement is presented as a poetic or literary expression — language that uses figurative or evocative form rather than literal assertion.

Statement: "{UNIT}"

Under this poetic-expression frame, provide your response as JSON with the following keys:
- communicative_function: what the statement is doing communicatively
- implicit_claims: any claims the statement implies or presupposes (list)
- coherence_assessment: brief assessment of internal coherence
- conditions_for_usefulness: what would have to be true for this statement to be useful
- conditions_for_misleading: what would have to be true for this statement to be misleading

Respond in JSON only, no preamble.'''
    },
    
    'F5_ai_ethics': {
        'name': 'AI ethics / human blueprint',
        'description': 'Frames the unit as guidance for AI system designers about human values to preserve.',
        'requires_author_metadata': False,
        'prompt_template': '''The following statement is presented as guidance for AI system designers — a principle about human values that AI systems should help preserve or protect.

Statement: "{UNIT}"

Under this AI-ethics frame, provide your response as JSON with the following keys:
- communicative_function: what the statement is doing communicatively
- implicit_claims: any claims the statement implies or presupposes (list)
- coherence_assessment: brief assessment of internal coherence
- conditions_for_usefulness: what would have to be true for this statement to be useful
- conditions_for_misleading: what would have to be true for this statement to be misleading

Respond in JSON only, no preamble.'''
    },
    
    'F6_author_named': {
        'name': 'Author identity included',
        'description': 'Same neutral framing as F0 but with author and source identified.',
        'requires_author_metadata': True,
        'prompt_template': '''The following statement is from "{TITLE}" by {AUTHOR}.

Statement: "{UNIT}"

Knowing the author and source, provide your response as JSON with the following keys:
- communicative_function: what the statement is doing communicatively
- implicit_claims: any claims the statement implies or presupposes (list)
- coherence_assessment: brief assessment of internal coherence
- conditions_for_usefulness: what would have to be true for this statement to be useful
- conditions_for_misleading: what would have to be true for this statement to be misleading

Respond in JSON only, no preamble.'''
    },
    
    'F7_author_anonymous': {
        'name': 'Author identity removed',
        'description': 'Same neutral framing as F0 with explicit note that author identity is withheld.',
        'requires_author_metadata': False,
        'prompt_template': '''The following statement is from an unspecified source. The author and context are not provided.

Statement: "{UNIT}"

Without knowing author or context, provide your response as JSON with the following keys:
- communicative_function: what the statement is doing communicatively
- implicit_claims: any claims the statement implies or presupposes (list)
- coherence_assessment: brief assessment of internal coherence
- conditions_for_usefulness: what would have to be true for this statement to be useful
- conditions_for_misleading: what would have to be true for this statement to be misleading

Respond in JSON only, no preamble.'''
    }
}

print(f'Defined {len(FRAMES)} frames:')
for fid, frame in FRAMES.items():
    print(f'  {fid}: {frame["name"]}')

## Section 4: Hash each frame for reproducibility

Each prompt template gets a SHA256 hash. The hash anchors the prompt to a specific wording. Any prompt change creates a new frame version with a new hash.

In [ ]:
for fid, frame in FRAMES.items():
    prompt_bytes = frame['prompt_template'].encode('utf-8')
    frame['prompt_sha256'] = hashlib.sha256(prompt_bytes).hexdigest()
    frame['prompt_sha256_short'] = frame['prompt_sha256'][:16]
    frame['frame_version'] = 'v1'

print(f'{"frame_id":24} {"version":8} {"sha256 (first 16)":>20}')
print('-' * 60)
for fid, frame in FRAMES.items():
    print(f'{fid:24} {frame["frame_version"]:8} {frame["prompt_sha256_short"]:>20}')

## Section 5: Render previews

For each frame, render an example with the first sampled unit so you can see exactly what gets sent to models. If the prompts read wrong here, revise before Stage 5 pilot.

In [ ]:
def render_prompt(frame, unit, text_metadata=None):
    """Render a frame prompt for a specific unit. text_metadata required for F6."""
    prompt = frame['prompt_template'].replace('{UNIT}', unit['unit_text'])
    if frame['requires_author_metadata']:
        if text_metadata is None:
            raise ValueError(f'Frame {frame["name"]} requires text metadata')
        prompt = prompt.replace('{TITLE}', text_metadata['title'])
        prompt = prompt.replace('{AUTHOR}', text_metadata['author'])
    return prompt

# Use the first sampled unit from C1 for previews
preview_unit = next(u for u in sampled_units if u['text_id'] == 'C1')
preview_text_meta = next(t for t in corpus['texts'] if t['text_id'] == 'C1')

print(f'Preview unit: [{preview_unit["unit_id"]}]')
print(f'Text: {preview_unit["unit_text"]}\n')

for fid, frame in FRAMES.items():
    print(f'\n{"=" * 70}')
    print(f'{fid}: {frame["name"]}')
    print(f'SHA256: {frame["prompt_sha256_short"]}')
    print(f'{"=" * 70}')
    rendered = render_prompt(frame, preview_unit, preview_text_meta)
    print(rendered)

## Section 6: Build the execution plan

Generate the full list of (unit, frame, model) tuples that will be executed in Stage 6. This is the canonical execution manifest — every API call is one row in this list.

In [ ]:
# Council models for the study
COUNCIL_MODELS = [
    {'model_id': 'claude_sonnet_4_6', 'provider': 'anthropic', 'api_model': 'claude-sonnet-4-6'},
    {'model_id': 'gpt_5', 'provider': 'openai', 'api_model': 'gpt-5'},
    {'model_id': 'gemini_2_5_pro', 'provider': 'google', 'api_model': 'gemini-2.5-pro'},
    {'model_id': 'grok_4', 'provider': 'xai', 'api_model': 'grok-4'},
    {'model_id': 'deepseek_v3', 'provider': 'deepseek', 'api_model': 'deepseek-chat'},
]

# Build the execution plan: every (unit, frame, model) triple
execution_plan = []

for unit in sampled_units:
    text_meta = next(t for t in corpus['texts'] if t['text_id'] == unit['text_id'])
    
    for frame_id, frame in FRAMES.items():
        # Render the prompt for this unit + frame
        meta_required = frame['requires_author_metadata']
        rendered_prompt = render_prompt(frame, unit, text_meta if meta_required else None)
        
        for model in COUNCIL_MODELS:
            execution_plan.append({
                'call_id': f'{unit["unit_id"]}__{frame_id}__{model["model_id"]}',
                'unit_id': unit['unit_id'],
                'text_id': unit['text_id'],
                'frame_id': frame_id,
                'frame_version': frame['frame_version'],
                'prompt_sha256': frame['prompt_sha256'],
                'model_id': model['model_id'],
                'provider': model['provider'],
                'api_model': model['api_model'],
                'rendered_prompt': rendered_prompt,
                'unit_text': unit['unit_text']
            })

print(f'Execution plan built: {len(execution_plan)} API calls total')
print(f'  {len(sampled_units)} units × {len(FRAMES)} frames × {len(COUNCIL_MODELS)} models = {len(sampled_units) * len(FRAMES) * len(COUNCIL_MODELS)}')

# Verify the count matches
assert len(execution_plan) == len(sampled_units) * len(FRAMES) * len(COUNCIL_MODELS)
print(f'Verification passed.')

In [ ]:
# Quick breakdown of execution plan distribution
from collections import Counter

by_text = Counter(c['text_id'] for c in execution_plan)
by_frame = Counter(c['frame_id'] for c in execution_plan)
by_model = Counter(c['model_id'] for c in execution_plan)

print('Calls per text:')
for text_id, count in sorted(by_text.items()):
    print(f'  {text_id}: {count}')
print('\nCalls per frame:')
for frame_id, count in sorted(by_frame.items()):
    print(f'  {frame_id}: {count}')
print('\nCalls per model:')
for model_id, count in sorted(by_model.items()):
    print(f'  {model_id}: {count}')

## Section 7: Freeze Stage 3 design artifact

Save the complete Stage 3 design — frames, sample, execution plan — as a single hashed artifact that Stage 5 (pilot) and Stage 6 (full execution) consume.

In [ ]:
stage3_design = {
    'stage': '3',
    'creation_date': datetime.now(timezone.utc).isoformat(),
    'corpus_version': CORPUS_VERSION,
    'corpus_sha256': CORPUS_SHA,
    'sample_per_text_target': SAMPLE_PER_TEXT,
    'sample_random_seed': SAMPLE_RANDOM_SEED,
    'frames': FRAMES,
    'council_models': COUNCIL_MODELS,
    'sample_meta': sample_meta,
    'sampled_units': sampled_units,
    'total_api_calls_planned': len(execution_plan),
    'execution_plan_summary': {
        'n_units': len(sampled_units),
        'n_frames': len(FRAMES),
        'n_models': len(COUNCIL_MODELS),
        'total_calls': len(execution_plan)
    }
}

# Compute SHA256 of the design (excluding the execution plan itself, which is derived)
design_serialized = json.dumps(stage3_design, sort_keys=True, ensure_ascii=False)
design_sha = hashlib.sha256(design_serialized.encode('utf-8')).hexdigest()
stage3_design['design_sha256'] = design_sha

design_path = f'{PROJECT_ROOT}/stage3/stage3_design_{design_sha[:16]}.json'
with open(design_path, 'w', encoding='utf-8') as f:
    json.dump(stage3_design, f, indent=2, ensure_ascii=False)

print(f'Stage 3 design frozen.')
print(f'Path: {design_path}')
print(f'SHA256: {design_sha}')

In [ ]:
# Save the execution plan as a separate file (it's large)
execution_plan_path = f'{PROJECT_ROOT}/stage3/execution_plan_{design_sha[:16]}.json'
with open(execution_plan_path, 'w', encoding='utf-8') as f:
    json.dump({
        'design_sha256': design_sha,
        'total_calls': len(execution_plan),
        'calls': execution_plan
    }, f, indent=2, ensure_ascii=False)

print(f'Execution plan saved: {execution_plan_path}')
print(f'File size: {os.path.getsize(execution_plan_path) / 1024:.1f} KB')
print(f'Total API calls: {len(execution_plan)}')

In [ ]:
# Human-readable manifest for the methods section
manifest = {
    'stage3_id': f'MTCA-1-stage3-{design_sha[:16]}',
    'creation_date': stage3_design['creation_date'],
    'design_sha256': design_sha,
    'corpus_version': CORPUS_VERSION,
    'corpus_sha256': CORPUS_SHA,
    'sample': {
        'target_per_text': SAMPLE_PER_TEXT,
        'random_seed': SAMPLE_RANDOM_SEED,
        'total_units': len(sampled_units),
        'breakdown': [
            {
                'text_id': m['text_id'],
                'lineage': m['lineage'],
                'sampled': m['units_sampled'],
                'available': m['units_in_corpus'],
                'method': m['sampling_method']
            } for m in sample_meta
        ]
    },
    'frames': [
        {
            'frame_id': fid,
            'name': frame['name'],
            'description': frame['description'],
            'version': frame['frame_version'],
            'sha256': frame['prompt_sha256']
        } for fid, frame in FRAMES.items()
    ],
    'council_models': COUNCIL_MODELS,
    'execution': {
        'total_api_calls': len(execution_plan),
        'budget_estimate_usd': '25-50',
        'wall_clock_estimate_hours': '3-6 unattended'
    }
}

manifest_path = f'{PROJECT_ROOT}/outputs/stage3_manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print(f'Stage 3 manifest saved: {manifest_path}')
print('\n' + json.dumps(manifest, indent=2))

## Section 8: Stage 3 closeout

**Frozen artifacts:**
- `stage3/stage3_design_{hash}.json` — frames, sample, council models, all hashed
- `stage3/execution_plan_{hash}.json` — every API call as a row, ready for Stage 6 execution
- `outputs/stage3_manifest.json` — human-readable summary for the methods section

**Reproducibility chain:**
- Corpus v1.1 SHA → Stage 3 design SHA → Stage 6 results will hash against this design
- Anyone can regenerate the exact same sample by running with the same seed and corpus
- Anyone can verify a prompt matches the frozen version by hashing it

**Stage 5 prerequisites met:**
- Eight frames defined and frozen
- Fifty-four units sampled deterministically
- Five council models specified
- Execution plan totals 2,160 API calls
- Budget estimate $25-50 across providers

**What Stage 5 does (next):**
Stage 5 is the pilot — runs one unit through one frame through all five models. Maybe 8-10 API calls total. Verifies the wiring works, the prompts produce parseable JSON, the storage structure handles responses cleanly. After pilot validates, Stage 6 runs the full 2,160-call execution.

**If you want to revise anything before Stage 5:**
- Change frame wording in Section 3, re-run from Section 3 onward, get new design SHA
- Change sample seed in Section 2, re-run, get new sample + new design SHA
- Add or remove frames, re-run, get new design SHA
- Add or remove council models, re-run, get new design SHA

The old design artifact stays on disk. Each revision creates a new one.